# Redis Chat Message History

This notebook demonstrates how to use the `RedisChatMessageHistory` class from the langchain-redis package to store and manage chat message history using Redis.

## Setup

First, we need to install the required dependencies and ensure we have a Redis instance running.

In [ ]:
!pip install -U langchain-redis langchain-openai redis

Make sure you have a Redis server running. You can start one using Docker with the following command:

```
docker run -d -p 6379:6379 redis:latest
```

Or install and run Redis locally according to the instructions for your operating system.

## Importing Required Libraries

In [ ]:
from langchain_redis import RedisChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
import os

## Basic Usage of RedisChatMessageHistory

In [ ]:
# Initialize RedisChatMessageHistory
history = RedisChatMessageHistory("user_123", url="redis://localhost:6379")

# Add messages to the history
history.add_user_message("Hello, AI assistant!")
history.add_ai_message("Hello! How can I assist you today?")

# Retrieve messages
print("Chat History:")
for message in history.messages:
    print(f"{type(message).__name__}: {message.content}")

## Using RedisChatMessageHistory with Language Models

In [ ]:
# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Initialize the language model
llm = ChatOpenAI()

# Create the conversational chain
chain = prompt | llm

# Function to get or create a RedisChatMessageHistory instance
def get_redis_history(session_id: str) -> BaseChatMessageHistory:
    return RedisChatMessageHistory(session_id, url="redis://localhost:6379")

# Create a runnable with message history
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_redis_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Use the chain in a conversation
response1 = chain_with_history.invoke(
    {"input": "Hi, my name is Alice."},
    config={"configurable": {"session_id": "alice_123"}}
)
print("AI Response 1:", response1.content)

response2 = chain_with_history.invoke(
    {"input": "What's my name?"},
    config={"configurable": {"session_id": "alice_123"}}
)
print("AI Response 2:", response2.content)

## Advanced Features

### Custom Redis Configuration

In [ ]:
# Initialize with custom Redis configuration
custom_history = RedisChatMessageHistory(
    "user_456",
    url="redis://localhost:6379",
    key_prefix="custom_prefix:",
    ttl=3600,  # Set TTL to 1 hour
    index_name="custom_index"
)

custom_history.add_user_message("This is a message with custom configuration.")
print("Custom History:", custom_history.messages)

### Searching Messages

In [ ]:
# Add more messages
history.add_user_message("Tell me about artificial intelligence.")
history.add_ai_message("Artificial Intelligence (AI) is a branch of computer science...")

# Search for messages containing a specific term
search_results = history.search_messages("artificial intelligence")
print("Search Results:")
for result in search_results:
    print(f"{result['type']}: {result['content'][:50]}...")

### Clearing History

In [ ]:
# Clear the chat history
history.clear()
print("Messages after clearing:", history.messages)

## Conclusion

This notebook demonstrated the key features of `RedisChatMessageHistory` from the langchain-redis package. It showed how to initialize and use the chat history, integrate it with language models, and utilize advanced features like custom configurations and message searching. Redis provides a fast and scalable solution for managing chat history in AI applications.